# ClarifyRL — GRPO Training with TRL

Train **Qwen3-1.7B** to ask clarifying questions before acting, using GRPO via TRL's `environment_factory` pattern.

- **Env**: ClarifyRL HF Space (`agarwalanu3103-clarify-rl`)
- **Compute**: T4 16GB (Colab free tier)
- **Time**: ~20min for 100 steps

In [1]:
!pip install -Uq "vllm>=0.12,<0.19"
!pip install -Uq "git+https://github.com/huggingface/trl.git@main" "git+https://github.com/huggingface/transformers.git@main" trackio requests
!pip uninstall -y wandb

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
trackio 0.25.0 requires huggingface-hub<2,>=1.10.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 5.50.0 requires gradio-client==1.14.0, but you have gradio-client 2.5.0 which is incompatible.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
vllm 0.18.1 requires transformers<5,>=4.56.0, but you have transformers 5.7.0.dev0 which is incompatible.
gradio 5.50.0 requires gradio-client==1.14.0, but you have gr

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

## Configuration

In [ ]:
import random

ENV_BASE_URL = "https://agarwalanu3103-clarify-rl.hf.space"
MODEL_NAME = "Qwen/Qwen3-1.7B"
OUTPUT_DIR = "clarify-rl-grpo-qwen3-1.7b"
MAX_STEPS = 100
SEED = 42

DIFFICULTIES = ["easy", "medium", "hard"]
DIFFICULTY_WEIGHTS = [0.5, 0.3, 0.2]

random.seed(SEED)

prompt = """You are a helpful assistant that books and plans things for users.
When you receive a request, you may not have all the information needed.
You can:
1. ASK clarifying questions using the ask_question tool (max 6 total)
2. PROPOSE a final plan using the propose_plan tool when you have enough info
3. GET the task description again using the get_task_info tool

Be efficient: ask only what you NEED, then propose a plan.
Do NOT include preferences in the plan that you weren't told about."""

## ClarifyEnv client

Wraps the HF Space into the interface TRL's `environment_factory` expects. Public methods with docstrings become tools the model can call.

In [ ]:
import json
import requests


class ClarifyEnv:
    def __init__(self):
        self.base_url = ENV_BASE_URL
        self.reward = 0.0
        self.done = False
        self._session = requests.Session()

    def reset(self, **kwargs) -> str | None:
        task_id = random.choices(DIFFICULTIES, weights=DIFFICULTY_WEIGHTS, k=1)[0]
        resp = self._session.post(f"{self.base_url}/reset", json={"task_id": task_id}, timeout=30)
        data = resp.json()
        obs = data.get("observation", {})
        self.reward = data.get("reward", 0.0)
        self.done = data.get("done", False)
        result = obs.get("result", {})
        if isinstance(result, str):
            result = json.loads(result)
        request_text = result.get("request", "")
        family = result.get("family", "")
        max_steps = result.get("max_steps", 10)
        return (
            f"USER REQUEST: {request_text}\n"
            f"Task family: {family}\n"
            f"You have {max_steps} steps. Ask clarifying questions, then propose a plan."
        )

    def _step(self, tool_name: str, arguments: dict) -> str:
        action = {"type": "call_tool", "tool_name": tool_name, "arguments": arguments}
        resp = self._session.post(f"{self.base_url}/step", json=action, timeout=30)
        data = resp.json()
        obs = data.get("observation", {})
        self.reward = data.get("reward", 0.0)
        self.done = data.get("done", False)
        result = obs.get("result", {})
        if isinstance(result, dict):
            return json.dumps(result)
        return str(result)

    def ask_question(self, question: str) -> str:
        """
        Ask a clarifying question to gather missing information from the user.

        Args:
            question: The clarifying question to ask.

        Returns:
            The user's response to your question.
        """
        return self._step("ask_question", {"question": question})

    def propose_plan(self, plan: str) -> str:
        """
        Submit your final plan as a JSON string with the required fields.

        Args:
            plan: A JSON string containing the plan fields you've gathered.

        Returns:
            The scoring result and episode outcome.
        """
        return self._step("propose_plan", {"plan": plan})

    def get_task_info(self) -> str:
        """
        Retrieve the current task description and requirements.

        Returns:
            The task information including the user request and instructions.
        """
        return self._step("get_task_info", {})

: 

## Smoke test: verify env reachable

Before training, confirm the HF Space responds. If this fails, training will fail too — fix the env first.

In [ ]:
_env = ClarifyEnv()
_obs = _env.reset()
print("OK — env reachable")
print("Initial observation:", _obs)
print("Reward:", _env.reward, "Done:", _env.done)
del _env, _obs

In [ ]:
## Reward function

Each rollout's reward is read from the `ClarifyEnv` instance after the episode ends.

def reward_func(environments, **kwargs) -> list[float]:
    return [env.reward for env in environments]

## Dataset

Each entry triggers one rollout episode. The prompt is the system instructions; the env supplies the actual user request via `reset()`.

In [ ]:
from datasets import Dataset

dataset = Dataset.from_dict({
    "prompt": [[{"role": "user", "content": prompt}] for _ in range(3000)]
})
print(f"Dataset: {len(dataset)} examples")

In [5]:
## GRPOConfig

vLLM colocated mode for fast generation on a single T4. Gradient checkpointing keeps memory low.

Initial observation: USER REQUEST: I have a problem.
Task family: medical_intake
You have 10 steps. Ask clarifying questions, then propose a plan.

Q1 response: {}
Reward: 0.0 Done: False

Plan response: {}
Reward: 0.0 Done: False


from trl import GRPOConfig

grpo_config = GRPOConfig(
    num_train_epochs=1,
    learning_rate=1e-6,
    gradient_accumulation_steps=16,
    per_device_train_batch_size=1,
    warmup_steps=10,
    max_steps=MAX_STEPS,
    optim="adamw_torch",
    max_grad_norm=1.0,
    num_generations=2,
    max_completion_length=1024,
    log_completions=True,
    num_completions_to_print=2,
    chat_template_kwargs={"enable_thinking": False},
    use_vllm=True,
    vllm_mode="colocate",
    vllm_gpu_memory_utilization=0.15,
    vllm_max_model_length=3072,
    output_dir=OUTPUT_DIR,
    report_to="trackio",
    trackio_space_id=OUTPUT_DIR,
    logging_steps=1,
    save_steps=50,
    save_total_limit=2,
    gradient_checkpointing=True,
    push_to_hub=True,
)

In [ ]:
## Initialize trainer

The `OutStream.fileno` patch is required: vLLM's `suppress_stdout` calls `sys.stdout.fileno()`, which Jupyter's ipykernel `OutStream` raises on by default. Patching the class to delegate to `sys.__stdout__` fixes it.

import sys
from ipykernel.iostream import OutStream
OutStream.fileno = lambda self: sys.__stdout__.fileno()

from trl import GRPOTrainer

trainer = GRPOTrainer(
    model=MODEL_NAME,
    reward_funcs=reward_func,
    train_dataset=dataset,
    args=grpo_config,
    environment_factory=ClarifyEnv,
)

In [ ]:
## Train

trainer_stats = trainer.train()

In [ ]:
## Save and push to Hub

trainer.save_model(OUTPUT_DIR)
trainer.push_to_hub()

In [ ]:
<!-- empty -->

TypeError: GRPOTrainer.__init__() missing 1 required positional argument: 'reward_funcs'

: 

<!-- empty -->

In [16]:
<!-- empty -->

GPU = Tesla T4. Max memory = 14.563 GB.
0.002 GB of memory reserved.


<!-- empty -->

In [12]:
<!-- empty -->

NameError: name 'trainer' is not defined

<!-- empty -->

In [ ]:
<!-- empty -->

: 

<!-- empty -->

In [ ]:
<!-- empty -->

: 

<!-- empty -->

In [ ]:
<!-- empty -->

: 